# Index coverage as a controlled variable in lexical vs dense code retrieval

Results notebook. It loads committed result files from the project repository and rebuilds every table and figure in the writeup. It runs no retrieval and needs no internet beyond cloning the repository.

Repository: https://github.com/pupa3066/context-configuration-builder

Design in one paragraph. A lexical retriever (BM25) searches every chunk of a corpus. A dense retriever indexes only a budget of chunks. When the budget is small, the two are compared on unequal ground. Here the dense index budget is varied directly while the corpus, questions and metrics stay fixed. BM25 is an internal control: its score cannot depend on the dense budget, so it must stay constant across budgets within a run.

Metrics: m1 is substring match of the target symbol in the retrieved chunks (favors lexical). m2 requires the symbol on a definition line (neither favored). m3 is m1 or a cosine credit above a threshold (favors dense).


In [ ]:
import json, os, glob, subprocess, platform
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

REF = 'feat/agent-track-submission'
subprocess.run('rm -rf /tmp/cck && git clone --depth=1 --branch ' + REF +
               ' https://github.com/pupa3066/context-configuration-builder.git /tmp/cck', shell=True, check=True)
HEAD = subprocess.run('git -C /tmp/cck rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
BM = '/tmp/cck/benchmark'
M = ('m1_substring', 'm2_defline', 'm3_sem_credit')

# Only these single-run sweep files are read. Add a file name here only after confirming it was
# produced by the committed runner (conceptual_fair_metrics.py) on the full N=240 question set.
ALLOW_SWEEPS = ['SWEEP_cap_1000.json', 'SWEEP_cap_100000.json']
# Threshold sensitivity runs (full corpus, committed runner, N=240), reported in their own table.
ALLOW_THRESH = ['SWEEP_threshold_0.65_N240.json', 'SWEEP_threshold_0.80_N240.json']

print('Environment: python', platform.python_version(), '| pandas', pd.__version__, '| numpy', np.__version__)
print('Repository commit read by this notebook:', HEAD)


In [ ]:
def fmt(x):
    return f'{x[0]:.3f} [{x[1]:.3f}, {x[2]:.3f}]'
def overlap(a, b):
    return a[1] <= b[2] and b[1] <= a[2]


## Table 1. Cap sweep, committed runner, bge-small-en-v1.5, k=6, N=240

Each cell is the mean with a 95% bootstrap interval (1,000 resamples). `overlap` means the two intervals overlap; it shows no detected difference, not equivalence.


In [ ]:
rows = []
for f in ALLOW_SWEEPS:
    p = os.path.join(BM, f)
    if not os.path.exists(p):
        print('missing:', f); continue
    d = json.load(open(p))
    assert d['N'] == 240, f"{f}: N={d['N']}, expected 240"
    for m in M:
        b, s = d['arms']['bm25'][m], d['arms']['semantic'][m]
        rows.append(dict(cap=d['max_chunks'], model=d['embed_model'], metric=m,
                         bm25=fmt(b), semantic=fmt(s), diff=round(b[0] - s[0], 4), overlap=overlap(b, s)))
t1 = pd.DataFrame(rows).sort_values(['metric', 'cap'])
display(t1) if len(t1) else print('no sweep files found')

# Internal control: BM25 must be identical across caps within this run.
if len(t1):
    for m in M:
        vals = t1[t1.metric == m]['bm25'].unique()
        print(m, 'BM25 constant across caps:', len(vals) == 1)


## Table 1b. m3 threshold sensitivity at full corpus (bge-small-en-v1.5, k=6, N=240)

The cosine threshold changes only m3; m1 and m2 are listed as a consistency check and should equal the full-corpus rows of Table 1. The 0.72 row is from `SWEEP_cap_100000.json`.


In [ ]:
rows = []
for f in ['SWEEP_cap_100000.json'] + ALLOW_THRESH:
    p = os.path.join(BM, f)
    if not os.path.exists(p): print('missing:', f); continue
    d = json.load(open(p)); assert d['N'] == 240 and d['max_chunks'] == 100000, f
    for m in M:
        b, s = d['arms']['bm25'][m], d['arms']['semantic'][m]
        rows.append(dict(threshold=d['sem_threshold'], metric=m, bm25=fmt(b), semantic=fmt(s),
                         diff=round(b[0] - s[0], 4), overlap=overlap(b, s)))
display(pd.DataFrame(rows).sort_values(['metric', 'threshold']))


## Table 2. Replication against the earlier run

The earlier run used the same code and question miner on an earlier clone of the three repositories (psf/requests, pallets/click, psf/black). Repository contents may have changed between clones, so small shifts are expected and the two runs are reported separately. Earlier values are from `benchmark/CURRENT_STATE.md` in the repository.


In [ ]:
EARLIER = {
    1000:   {'bm25': (0.8667, 0.8208, 0.9583), 'semantic': (0.4417, 0.2792, 0.7125)},
    100000: {'bm25': (0.8667, 0.8208, 0.9583), 'semantic': (0.8667, 0.8125, 0.9333)},
}
rows = []
for f in ALLOW_SWEEPS:
    p = os.path.join(BM, f)
    if not os.path.exists(p): continue
    d = json.load(open(p)); cap = d['max_chunks']
    for arm in ('bm25', 'semantic'):
        for i, m in enumerate(M):
            new = d['arms'][arm][m][0]; old = EARLIER[cap][arm][i]
            rows.append(dict(cap=cap, arm=arm, metric=m, earlier=old, this_run=new, delta=round(new - old, 4)))
display(pd.DataFrame(rows)) if rows else print('no data')


## Table 3. Chunk size as a second controlled factor

`confound_sweep.py` imports the committed question miner, BM25, chunker, retriever and metrics, and changes only the chunker (8 lines is the committed default). It saves per-question hits, so the BM25 minus dense difference is tested pairwise: a paired bootstrap interval and an exact McNemar test on the discordant questions.


In [ ]:
rows = []
files = sorted(glob.glob(os.path.join(BM, 'CONFOUND_*.json')))
for p in files:
    d = json.load(open(p))
    if d['N'] != 240:
        print('skipping (N != 240):', os.path.basename(p)); continue
    frac = np.mean(list(d['indexed_fraction'].values()))
    ts = d.get('chunk_token_stats', {})
    for m in M:
        b, s = d['summary']['arms']['bm25'][m], d['summary']['arms']['semantic'][m]
        pr = d['summary']['paired'][m]
        rows.append(dict(repos='/'.join(d['n_chunks']), chunker=d['chunker'], dense_mode=d.get('sem_mode', 'whole'), embedder=d['embed_model'].split('/')[-1], cap=d['cap'], indexed_fraction=round(float(frac), 3),
                         chunks_over_limit=ts.get('frac_over_limit'), metric=m,
                         bm25=fmt(b), dense=fmt(s), paired_diff=pr['mean_diff_bm25_minus_sem'],
                         paired_ci=pr['ci95'], bm25_only=pr['bm25_only'], dense_only=pr['sem_only'],
                         mcnemar_p=pr['mcnemar_exact_p']))
t3 = pd.DataFrame(rows)
if len(t3):
    display(t3.sort_values(['metric', 'repos', 'chunker', 'dense_mode', 'cap']))
else:
    print('no CONFOUND_*.json files in the repository yet')


In [ ]:
if len(t3):
    sub = t3[(t3.metric == 'm1_substring') & (t3.repos == 'requests/click/black') & (t3.dense_mode == 'whole') & (t3.embedder == 'bge-small-en-v1.5')].copy()
    sub['label'] = sub['chunker'] + '\ncap ' + sub['cap'].astype(str) + '\n(' + (sub['indexed_fraction'] * 100).round(0).astype(int).astype(str) + '% indexed)'
    def mean_of(s): return float(s.split(' ')[0])
    sub['b'] = sub['bm25'].map(mean_of); sub['s'] = sub['dense'].map(mean_of)
    sub = sub.sort_values(['chunker', 'cap'])
    x = np.arange(len(sub))
    fig, ax = plt.subplots(figsize=(max(7, 1.5 * len(sub)), 4.2))
    ax.bar(x - 0.18, sub['b'], 0.36, label='BM25 (all chunks)', color='#475569')
    ax.bar(x + 0.18, sub['s'], 0.36, label='Dense (budgeted index)', color='#2563eb')
    ax.set_xticks(x); ax.set_xticklabels(sub['label'], fontsize=8)
    ax.set_ylabel('m1 substring hit rate'); ax.set_ylim(0, 1.05)
    ax.set_title('m1 by chunker and dense index budget (N=240, k=6)')
    ax.legend(); ax.grid(axis='y', alpha=0.3); fig.tight_layout()
    fig.savefig('/kaggle/working/fig_m1_chunker_cap.png', dpi=140)
    print('saved /kaggle/working/fig_m1_chunker_cap.png')
else:
    print('figure skipped: no confound results yet')


In [ ]:
# Validity checks printed for the reader.
print('1. N=240 asserted for every file used.')
if len(t3):
    for key, g in t3[t3.metric == 'm1_substring'].groupby(['repos', 'chunker']):
        print('2. BM25 m1 constant across caps for', key, ':', g['bm25'].nunique() == 1)
print('3. All files come from repository commit', HEAD)


## Scope

- 240 questions mined from docstrings of three Python repositories. Docstring-mined questions share vocabulary with the target text, which favors lexical matching. Human-written queries, other languages and code-specialized embedders were not tested here.
- m1 and m3 reward retrieving the target symbol, not solving a task. These results do not measure agent task success.
- Overlapping intervals show no detected difference, not equivalence. Table 3 adds paired tests for that reason.
- Only the files listed in the setup cell are read. A result is included only if it came from the committed code.
